# Лабораторная работа №3: классификация. Деревья решений

**Автор:** Гроза И.В.


## Содержание

1. Анализ постановки задачи
2. План выполнения
3. Описание датасета `weatherAUS.csv`
4. Считывание и подготовка данных
5. `DecisionTreeClassifier`
6. `RandomForestClassifier`
7. `KNeighborsClassifier`
8. Общее подмножество признаков
9. Визуализация дерева решений
10. Сравнение моделей на тестовой выборке
11. Итоговые выводы


## Анализ постановки задачи

Для ЛР3 нужно решить одну задачу бинарной классификации на `weatherAUS.csv` с целевым признаком `RainTomorrow`.

Ключевые особенности задачи:

- в датасете есть пропуски, поэтому требуется явная очистка данных;
- есть числовые и категориальные признаки, значит нужно выполнить кодирование;
- классы несбалансированы, поэтому балансировка делается только на обучающей части;
- нужно применить `GridSearchCV` для трёх моделей: `DecisionTreeClassifier`, `RandomForestClassifier`, `KNeighborsClassifier`;
- для каждой модели требуется сравнить качество до и после фильтрации признаков;
- итоговое честное сравнение выполняется только на тестовой выборке.

В качестве основной метрики подбора используем `f1`, так как положительный класс `RainTomorrow = Yes` встречается заметно реже и важен баланс между `precision` и `recall`.


## План выполнения

1. Считать и описать `weatherAUS.csv`, показать пропуски и распределение целевого признака.
2. Очистить данные: удалить строки без `RainTomorrow`, разложить `Date` на календарные признаки, убрать столбцы с большим числом пропусков, заполнить оставшиеся пропуски.
3. Закодировать категориальные признаки, нормализовать небинарные признаки и разделить выборку на train/test.
4. Снизить дисбаланс классов на обучающей выборке простым downsampling мажоритарного класса.
5. Для `DecisionTreeClassifier`, `RandomForestClassifier` и `KNeighborsClassifier` подобрать гиперпараметры через `GridSearchCV`.
6. Для каждой модели выполнить фильтрацию признаков и повторный подбор гиперпараметров.
7. Если хотя бы одна модель улучшится после фильтрации, собрать общее подмножество признаков и переобучить все три модели на одном наборе признаков.
8. Визуализировать итоговое дерево решений и сравнить лучшие модели на тестовой выборке.


In [1]:
import os
import warnings
from pathlib import Path

os.environ["LOKY_MAX_CPU_COUNT"] = "1"
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
TEST_SIZE = 0.2
CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

root_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(path for path in root_candidates if (path / "data").exists())
DATA_DIR = PROJECT_ROOT / "data"

print(f"Project root: {PROJECT_ROOT}")
print("Target: RainTomorrow")


Project root: C:\Users\Марина\PycharmProjects\sem8-ait-ds
Target: RainTomorrow


In [2]:
def binary_metrics(y_true, y_pred):
    return pd.Series(
        {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
        }
    )


def downsample_binary(X, y, random_state=RANDOM_STATE):
    # Балансируем только обучающую выборку.
    data = X.copy()
    data["target"] = y.values
    class_counts = data["target"].value_counts()
    minority_class = class_counts.idxmin()
    minority_size = class_counts.min()

    sampled_parts = []
    for class_value, group in data.groupby("target"):
        if class_value == minority_class:
            sampled_parts.append(group)
        else:
            sampled_parts.append(group.sample(n=minority_size, random_state=random_state))

    balanced = (
        pd.concat(sampled_parts)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )
    return balanced.drop(columns="target"), balanced["target"]


def scale_non_binary_features(X_train, X_test):
    binary_columns = X_train.columns[X_train.isin([0, 1]).all()].tolist()
    scaled_columns = [col for col in X_train.columns if col not in binary_columns]

    scaler = StandardScaler()
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    X_train_scaled[scaled_columns] = scaler.fit_transform(X_train[scaled_columns])
    X_test_scaled[scaled_columns] = scaler.transform(X_test[scaled_columns])

    return X_train_scaled, X_test_scaled, scaled_columns, binary_columns


def fit_grid_search(estimator, param_grid, X_train, y_train, scoring="f1"):
    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring=scoring,
        cv=CV,
        n_jobs=1,
        refit=True,
    )
    grid.fit(X_train, y_train)
    return grid


def run_experiment(model_name, estimator, param_grid, X_fit, y_fit, X_eval, y_eval):
    grid = fit_grid_search(estimator, param_grid, X_fit, y_fit)
    best_model = grid.best_estimator_
    train_pred = best_model.predict(X_eval)
    return {
        "model_name": model_name,
        "grid": grid,
        "best_model": best_model,
        "train_metrics": binary_metrics(y_eval, train_pred),
    }


def display_search_result(result):
    print(f"Модель: {result['model_name']}")
    print(f"Лучшая средняя F1 на CV: {result['grid'].best_score_:.4f}")
    display(pd.Series(result["grid"].best_params_, name="value").to_frame("best_value"))
    display(result["train_metrics"].rename("value").to_frame())


def select_features(scores, min_features=20):
    scores = pd.Series(scores).sort_values(ascending=False)
    positive_scores = scores[scores > 0]

    if positive_scores.empty:
        selected = scores.head(min_features).index.tolist()
        threshold = float(scores.iloc[min_features - 1])
    else:
        threshold = float(positive_scores.median())
        selected = positive_scores[positive_scores >= threshold].index.tolist()
        if len(selected) < min_features:
            selected = scores.head(min_features).index.tolist()
            threshold = float(scores.loc[selected[-1]])

    selected = list(dict.fromkeys(selected))
    return selected, threshold, scores


def compare_metric_series(left_name, left_series, right_name, right_series):
    return pd.concat(
        [left_series.rename(left_name), right_series.rename(right_name)],
        axis=1,
    ).round(4)


def test_result_row(model_name, model, X_test, y_test):
    metrics = binary_metrics(y_test, model.predict(X_test))
    return pd.Series({"model": model_name, **metrics.to_dict()})


## Описание датасета `weatherAUS.csv`

Источник данных:
[Rain in Australia на Kaggle](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package).

Датасет содержит ежедневные метеорологические наблюдения по Австралии. Задача — предсказать, будет ли дождь завтра.

**Целевой признак:** `RainTomorrow`.

| Признак | Описание | Единицы |
|---|---|---|
| `Date` | Дата наблюдения | календарная дата |
| `Location` | Название метеостанции | категориальный |
| `MinTemp` | Минимальная температура за последние 24 часа к 9 утра | °C |
| `MaxTemp` | Максимальная температура за последние 24 часа от 9 утра | °C |
| `Rainfall` | Осадки за последние 24 часа к 9 утра | мм |
| `Evaporation` | Испарение за последние 24 часа | мм |
| `Sunshine` | Продолжительность солнечного сияния | часы |
| `WindGustDir` | Направление самого сильного порыва ветра | 16 румбов |
| `WindGustSpeed` | Скорость самого сильного порыва ветра | км/ч |
| `WindDir9am` | Направление ветра в 9 утра | румбы |
| `WindDir3pm` | Направление ветра в 3 часа дня | румбы |
| `WindSpeed9am` | Скорость ветра в 9 утра | км/ч |
| `WindSpeed3pm` | Скорость ветра в 3 часа дня | км/ч |
| `Humidity9am` | Относительная влажность в 9 утра | % |
| `Humidity3pm` | Относительная влажность в 3 часа дня | % |
| `Pressure9am` | Атмосферное давление в 9 утра | гПа |
| `Pressure3pm` | Атмосферное давление в 3 часа дня | гПа |
| `Cloud9am` | Облачность в 9 утра | восьмые доли неба |
| `Cloud3pm` | Облачность в 3 часа дня | восьмые доли неба |
| `Temp9am` | Температура в 9 утра | °C |
| `Temp3pm` | Температура в 3 часа дня | °C |
| `RainToday` | Был ли дождь сегодня | `Yes/No` |
| `RainTomorrow` | Будет ли дождь завтра | `Yes/No` |

В предобработке признак `Date` будет разложен на `Year`, `Month`, `Day`, чтобы сохранить сезонность и убрать строковое представление даты.


## Считывание и первичный анализ данных

Сначала посмотрим на размерность датафрейма, примеры строк, типы столбцов, пропуски и распределение целевого признака.


In [3]:
weather_raw = pd.read_csv(DATA_DIR / "weatherAUS.csv")

print(f"Размерность исходного датафрейма: {weather_raw.shape}")
display(weather_raw.head())
weather_raw.info()


Размерность исходного датафрейма: (145460, 23)


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,WNW,20.0,24.0,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,WSW,4.0,22.0,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,WSW,19.0,26.0,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,E,11.0,9.0,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,NW,7.0,20.0,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


<class 'pandas.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           145460 non-null  str    
 1   Location       145460 non-null  str    
 2   MinTemp        143975 non-null  float64
 3   MaxTemp        144199 non-null  float64
 4   Rainfall       142199 non-null  float64
 5   Evaporation    82670 non-null   float64
 6   Sunshine       75625 non-null   float64
 7   WindGustDir    135134 non-null  str    
 8   WindGustSpeed  135197 non-null  float64
 9   WindDir9am     134894 non-null  str    
 10  WindDir3pm     141232 non-null  str    
 11  WindSpeed9am   143693 non-null  float64
 12  WindSpeed3pm   142398 non-null  float64
 13  Humidity9am    142806 non-null  float64
 14  Humidity3pm    140953 non-null  float64
 15  Pressure9am    130395 non-null  float64
 16  Pressure3pm    130432 non-null  float64
 17  Cloud9am       89572 non-null   float64


In [4]:
weather_missing = (
    weather_raw.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

weather_target_distribution = (
    weather_raw["RainTomorrow"]
    .value_counts(dropna=False)
    .rename_axis("RainTomorrow")
    .reset_index(name="count")
)

display(weather_missing.head(15))
display(weather_target_distribution)


,missing_count
Sunshine,69835
Evaporation,62790
Cloud3pm,59358
Cloud9am,55888
Pressure9am,15065
Pressure3pm,15028
WindDir9am,10566
WindGustDir,10326
WindGustSpeed,10263
Humidity3pm,4507


,RainTomorrow,count
0,No,110316
1,Yes,31877
2,NaN,3267


## Подготовка данных

Приняты следующие решения по очистке и преобразованию:

- удалить строки, где отсутствует целевой признак `RainTomorrow`;
- разложить `Date` на `Year`, `Month`, `Day`;
- убрать признаки, где доля пропусков превышает `35%`;
- числовые признаки заполнить медианой, категориальные — модой;
- бинарные признаки `RainToday` и `RainTomorrow` перевести в `0/1`;
- категориальные признаки закодировать через `pd.get_dummies`;
- после разбиения на train/test нормализовать небинарные признаки и отдельно снизить дисбаланс только на train.


In [5]:
weather = weather_raw.dropna(subset=["RainTomorrow"]).copy()

weather["Date"] = pd.to_datetime(weather["Date"])
weather["Year"] = weather["Date"].dt.year
weather["Month"] = weather["Date"].dt.month
weather["Day"] = weather["Date"].dt.day
weather = weather.drop(columns=["Date"])

missing_share = weather.isna().mean().sort_values(ascending=False)
dropped_columns = missing_share[missing_share > 0.35].index.tolist()
weather = weather.drop(columns=dropped_columns)

categorical_columns = ["Location", "WindGustDir", "WindDir9am", "WindDir3pm", "RainToday"]
numeric_columns = [col for col in weather.columns if col not in categorical_columns + ["RainTomorrow"]]

for col in categorical_columns:
    weather[col] = weather[col].fillna(weather[col].mode()[0])

for col in numeric_columns:
    weather[col] = weather[col].fillna(weather[col].median())

weather["RainToday"] = weather["RainToday"].map({"No": 0, "Yes": 1}).astype(int)
weather["RainTomorrow"] = weather["RainTomorrow"].map({"No": 0, "Yes": 1}).astype(int)

weather_encoded = pd.get_dummies(
    weather,
    columns=["Location", "WindGustDir", "WindDir9am", "WindDir3pm"],
    drop_first=True,
    dtype=int,
)

print("Удалённые признаки по порогу пропусков > 35%:")
print(dropped_columns)
print()
print(f"Размерность после кодирования: {weather_encoded.shape}")
display(weather_encoded.head())


Удалённые признаки по порогу пропусков > 35%:
['Sunshine', 'Evaporation', 'Cloud3pm', 'Cloud9am']

Размерность после кодирования: (142193, 110)


,MinTemp,MaxTemp,Rainfall,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Temp9am,Temp3pm,RainToday,RainTomorrow,Year,Month,Day,Location_Albany,Location_Albury,Location_AliceSprings,Location_BadgerysCreek,Location_Ballarat,Location_Bendigo,Location_Brisbane,Location_Cairns,Location_Canberra,Location_Cobar,Location_CoffsHarbour,Location_Dartmoor,Location_Darwin,Location_GoldCoast,Location_Hobart,Location_Katherine,Location_Launceston,Location_Melbourne,Location_MelbourneAirport,Location_Mildura,Location_Moree,Location_MountGambier,Location_MountGinini,Location_Newcastle,Location_Nhil,Location_NorahHead,Location_NorfolkIsland,Location_Nuriootpa,Location_PearceRAAF,Location_Penrith,Location_Perth,Location_PerthAirport,Location_Portland,Location_Richmond,Location_Sale,Location_SalmonGums,Location_Sydney,Location_SydneyAirport,Location_Townsville,Location_Tuggeranong,Location_Uluru,Location_WaggaWagga,Location_Walpole,Location_Watsonia,Location_Williamtown,Location_Witchcliffe,Location_Wollongong,Location_Woomera,WindGustDir_ENE,WindGustDir_ESE,WindGustDir_N,WindGustDir_NE,WindGustDir_NNE,WindGustDir_NNW,WindGustDir_NW,WindGustDir_S,WindGustDir_SE,WindGustDir_SSE,WindGustDir_SSW,WindGustDir_SW,WindGustDir_W,WindGustDir_WNW,WindGustDir_WSW,WindDir9am_ENE,WindDir9am_ESE,WindDir9am_N,WindDir9am_NE,WindDir9am_NNE,WindDir9am_NNW,WindDir9am_NW,WindDir9am_S,WindDir9am_SE,WindDir9am_SSE,WindDir9am_SSW,WindDir9am_SW,WindDir9am_W,WindDir9am_WNW,WindDir9am_WSW,WindDir3pm_ENE,WindDir3pm_ESE,WindDir3pm_N,WindDir3pm_NE,WindDir3pm_NNE,WindDir3pm_NNW,WindDir3pm_NW,WindDir3pm_S,WindDir3pm_SE,WindDir3pm_SSE,WindDir3pm_SSW,WindDir3pm_SW,WindDir3pm_W,WindDir3pm_WNW,WindDir3pm_WSW
0,13.4,22.9,0.6,44.0,20.0,24.0,71.0,22.0,1007.7,1007.1,16.9,21.8,0,0,2008,12,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,7.4,25.1,0.0,44.0,4.0,22.0,44.0,25.0,1010.6,1007.8,17.2,24.3,0,0,2008,12,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,12.9,25.7,0.0,46.0,19.0,26.0,38.0,30.0,1007.6,1008.7,21.0,23.2,0,0,2008,12,3,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,9.2,28.0,0.0,24.0,11.0,9.0,45.0,16.0,1017.6,1012.8,18.1,26.5,0,0,2008,12,4,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,17.5,32.3,1.0,41.0,7.0,20.0,82.0,33.0,1010.8,1006.0,17.8,29.7,0,0,2008,12,5,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0


In [6]:
X = weather_encoded.drop(columns=["RainTomorrow"])
y = weather_encoded["RainTomorrow"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train_scaled, X_test_scaled, scaled_columns, binary_columns = scale_non_binary_features(X_train, X_test)
X_train_bal, y_train_bal = downsample_binary(X_train_scaled, y_train)

balance_table = pd.DataFrame(
    {
        "train_before": y_train.value_counts().sort_index(),
        "train_after_downsampling": y_train_bal.value_counts().sort_index(),
        "test": y_test.value_counts().sort_index(),
    }
)
balance_table.index = ["No (0)", "Yes (1)"]

print(f"Число признаков после кодирования: {X_train.shape[1]}")
print(f"Масштабируемых признаков: {len(scaled_columns)}")
print(f"Бинарных/one-hot признаков: {len(binary_columns)}")
display(balance_table)


Число признаков после кодирования: 109
Масштабируемых признаков: 15
Бинарных/one-hot признаков: 94


,train_before,train_after_downsampling,test
No (0),88252,25502,22064
Yes (1),25502,25502,6375


## Сетки гиперпараметров

Для полного датасета используем компактные сетки и `StratifiedKFold(n_splits=3)`, чтобы ноутбук оставался выполнимым без лишнего перебора.


In [7]:
decision_tree_grid = {
    "max_depth": [6, 10, 14],
    "min_samples_leaf": [1, 5, 10],
    "max_features": [None, "sqrt"],
}

random_forest_grid = {
    "n_estimators": [120, 200],
    "max_depth": [10, None],
    "min_samples_leaf": [1, 5],
    "max_features": ["sqrt", 0.5],
}

knn_grid = {
    "n_neighbors": [5, 11, 21, 31],
    "weights": ["uniform", "distance"],
}

grid_overview = pd.DataFrame(
    [
        {
            "model": "DecisionTreeClassifier",
            "grid_points": int(np.prod([len(v) for v in decision_tree_grid.values()])),
        },
        {
            "model": "RandomForestClassifier",
            "grid_points": int(np.prod([len(v) for v in random_forest_grid.values()])),
        },
        {
            "model": "KNeighborsClassifier",
            "grid_points": int(np.prod([len(v) for v in knn_grid.values()])),
        },
    ]
)
display(grid_overview)


,model,grid_points
0,DecisionTreeClassifier,18
1,RandomForestClassifier,16
2,KNeighborsClassifier,8


## `DecisionTreeClassifier`

Сначала подбираем лучшую конфигурацию дерева на всех признаках, затем оцениваем важности признаков и повторяем подбор на отфильтрованном наборе.


In [8]:
dt_all_result = run_experiment(
    model_name="DecisionTreeClassifier (all features)",
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=decision_tree_grid,
    X_fit=X_train_bal,
    y_fit=y_train_bal,
    X_eval=X_train_scaled,
    y_eval=y_train,
)
display_search_result(dt_all_result)


Модель: DecisionTreeClassifier (all features)
Лучшая средняя F1 на CV: 0.7502


,best_value
max_depth,10.0
max_features,NaN
min_samples_leaf,1.0


,value
accuracy,0.785906
precision,0.514496
recall,0.798879
f1,0.625899


In [9]:
dt_selected_features, dt_threshold, dt_importance = select_features(
    pd.Series(
        dt_all_result["best_model"].feature_importances_,
        index=X_train_bal.columns,
    ),
    min_features=20,
)

display(dt_importance.head(20).rename("importance").to_frame())
print(f"Порог важности: {dt_threshold:.6f}")
print(f"Оставлено признаков: {len(dt_selected_features)} из {X_train_bal.shape[1]}")
print("Первые 25 отобранных признаков:")
print(dt_selected_features[:25])


,importance
Humidity3pm,0.574474
WindGustSpeed,0.113517
Pressure3pm,0.086262
Rainfall,0.041839
Temp3pm,0.016785
Pressure9am,0.016547
MaxTemp,0.015887
MinTemp,0.014833
Humidity9am,0.012674
Temp9am,0.009601


Порог важности: 0.000611
Оставлено признаков: 46 из 109
Первые 25 отобранных признаков:
['Humidity3pm', 'WindGustSpeed', 'Pressure3pm', 'Rainfall', 'Temp3pm', 'Pressure9am', 'MaxTemp', 'MinTemp', 'Humidity9am', 'Temp9am', 'WindSpeed3pm', 'Month', 'WindSpeed9am', 'Year', 'WindDir9am_NNE', 'Day', 'Location_MountGinini', 'Location_Darwin', 'Location_Wollongong', 'Location_Albany', 'WindDir9am_N', 'WindGustDir_S', 'WindDir3pm_N', 'Location_NorahHead', 'WindDir3pm_SE']


In [10]:
dt_filtered_result = run_experiment(
    model_name="DecisionTreeClassifier (filtered features)",
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=decision_tree_grid,
    X_fit=X_train_bal[dt_selected_features],
    y_fit=y_train_bal,
    X_eval=X_train_scaled[dt_selected_features],
    y_eval=y_train,
)
display_search_result(dt_filtered_result)

dt_compare = compare_metric_series(
    "all_features",
    dt_all_result["train_metrics"],
    "filtered_features",
    dt_filtered_result["train_metrics"],
)
display(dt_compare)

better_dt_variant = (
    "filtered_features"
    if dt_filtered_result["train_metrics"]["f1"] > dt_all_result["train_metrics"]["f1"]
    else "all_features"
)
print(f"Вывод: по F1 на обучающей выборке лучше вариант '{better_dt_variant}'.")


Модель: DecisionTreeClassifier (filtered features)
Лучшая средняя F1 на CV: 0.7496


,best_value
max_depth,10.0
max_features,NaN
min_samples_leaf,1.0


,value
accuracy,0.784500
precision,0.512424
recall,0.798957
f1,0.624387


,all_features,filtered_features
accuracy,0.7859,0.7845
precision,0.5145,0.5124
recall,0.7989,0.7990
f1,0.6259,0.6244


Вывод: по F1 на обучающей выборке лучше вариант 'all_features'.


## `RandomForestClassifier`

Для случайного леса используем встроенные `feature_importances_` и повторяем ту же схему: подбор на полном наборе признаков, затем фильтрация и новый подбор.


In [ ]:
rf_all_result = run_experiment(
    model_name="RandomForestClassifier (all features)",
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    param_grid=random_forest_grid,
    X_fit=X_train_bal,
    y_fit=y_train_bal,
    X_eval=X_train_scaled,
    y_eval=y_train,
)
display_search_result(rf_all_result)


In [ ]:
rf_selected_features, rf_threshold, rf_importance = select_features(
    pd.Series(
        rf_all_result["best_model"].feature_importances_,
        index=X_train_bal.columns,
    ),
    min_features=20,
)

display(rf_importance.head(20).rename("importance").to_frame())
print(f"Порог важности: {rf_threshold:.6f}")
print(f"Оставлено признаков: {len(rf_selected_features)} из {X_train_bal.shape[1]}")
print("Первые 25 отобранных признаков:")
print(rf_selected_features[:25])


In [ ]:
rf_filtered_result = run_experiment(
    model_name="RandomForestClassifier (filtered features)",
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    param_grid=random_forest_grid,
    X_fit=X_train_bal[rf_selected_features],
    y_fit=y_train_bal,
    X_eval=X_train_scaled[rf_selected_features],
    y_eval=y_train,
)
display_search_result(rf_filtered_result)

rf_compare = compare_metric_series(
    "all_features",
    rf_all_result["train_metrics"],
    "filtered_features",
    rf_filtered_result["train_metrics"],
)
display(rf_compare)

better_rf_variant = (
    "filtered_features"
    if rf_filtered_result["train_metrics"]["f1"] > rf_all_result["train_metrics"]["f1"]
    else "all_features"
)
print(f"Вывод: по F1 на обучающей выборке лучше вариант '{better_rf_variant}'.")


## `KNeighborsClassifier`

Для `kNN` собственной встроенной важности признаков нет, поэтому для фильтрации используем `mutual_info_classif`. Поиск параметров всё равно делается через `GridSearchCV`.


In [ ]:
knn_all_result = run_experiment(
    model_name="KNeighborsClassifier (all features)",
    estimator=KNeighborsClassifier(n_jobs=1),
    param_grid=knn_grid,
    X_fit=X_train_bal,
    y_fit=y_train_bal,
    X_eval=X_train_scaled,
    y_eval=y_train,
)
display_search_result(knn_all_result)


In [ ]:
knn_selected_features, knn_threshold, knn_importance = select_features(
    pd.Series(
        mutual_info_classif(X_train_bal, y_train_bal, random_state=RANDOM_STATE),
        index=X_train_bal.columns,
    ),
    min_features=20,
)

display(knn_importance.head(20).rename("mutual_information").to_frame())
print(f"Порог информативности: {knn_threshold:.6f}")
print(f"Оставлено признаков: {len(knn_selected_features)} из {X_train_bal.shape[1]}")
print("Первые 25 отобранных признаков:")
print(knn_selected_features[:25])


In [ ]:
knn_filtered_result = run_experiment(
    model_name="KNeighborsClassifier (filtered features)",
    estimator=KNeighborsClassifier(n_jobs=1),
    param_grid=knn_grid,
    X_fit=X_train_bal[knn_selected_features],
    y_fit=y_train_bal,
    X_eval=X_train_scaled[knn_selected_features],
    y_eval=y_train,
)
display_search_result(knn_filtered_result)

knn_compare = compare_metric_series(
    "all_features",
    knn_all_result["train_metrics"],
    "filtered_features",
    knn_filtered_result["train_metrics"],
)
display(knn_compare)

better_knn_variant = (
    "filtered_features"
    if knn_filtered_result["train_metrics"]["f1"] > knn_all_result["train_metrics"]["f1"]
    else "all_features"
)
print(f"Вывод: по F1 на обучающей выборке лучше вариант '{better_knn_variant}'.")


## Общее подмножество признаков

Если после фильтрации признаков хотя бы одна модель улучшилась по `f1`, формируем общее подмножество как объединение успешных наборов признаков и заново обучаем все три модели уже на одном и том же наборе признаков.


In [ ]:
model_definitions = {
    "DecisionTreeClassifier": {
        "estimator": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "grid": decision_tree_grid,
        "all_result": dt_all_result,
        "filtered_result": dt_filtered_result,
        "filtered_features": dt_selected_features,
    },
    "RandomForestClassifier": {
        "estimator": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
        "grid": random_forest_grid,
        "all_result": rf_all_result,
        "filtered_result": rf_filtered_result,
        "filtered_features": rf_selected_features,
    },
    "KNeighborsClassifier": {
        "estimator": KNeighborsClassifier(n_jobs=1),
        "grid": knn_grid,
        "all_result": knn_all_result,
        "filtered_result": knn_filtered_result,
        "filtered_features": knn_selected_features,
    },
}

variant_overview = pd.DataFrame(
    [
        {
            "model": model_name,
            "f1_all_features": cfg["all_result"]["train_metrics"]["f1"],
            "f1_filtered_features": cfg["filtered_result"]["train_metrics"]["f1"],
        }
        for model_name, cfg in model_definitions.items()
    ]
).set_index("model")
display(variant_overview.round(4))

improved_feature_sets = [
    cfg["filtered_features"]
    for cfg in model_definitions.values()
    if cfg["filtered_result"]["train_metrics"]["f1"] > cfg["all_result"]["train_metrics"]["f1"]
]

use_common_subset = len(improved_feature_sets) > 0
final_results = {}
final_feature_sets = {}

if use_common_subset:
    common_features = sorted(set().union(*improved_feature_sets))
    print(
        "Фильтрация улучшила хотя бы одну модель, "
        f"поэтому строим общее подмножество из {len(common_features)} признаков."
    )

    for model_name, cfg in model_definitions.items():
        final_results[model_name] = run_experiment(
            model_name=f"{model_name} (common subset)",
            estimator=cfg["estimator"],
            param_grid=cfg["grid"],
            X_fit=X_train_bal[common_features],
            y_fit=y_train_bal,
            X_eval=X_train_scaled[common_features],
            y_eval=y_train,
        )
        final_feature_sets[model_name] = common_features
else:
    print("Ни одна модель не улучшилась после фильтрации. Пункт 6 пропускается.")
    for model_name, cfg in model_definitions.items():
        if cfg["filtered_result"]["train_metrics"]["f1"] > cfg["all_result"]["train_metrics"]["f1"]:
            final_results[model_name] = cfg["filtered_result"]
            final_feature_sets[model_name] = cfg["filtered_features"]
        else:
            final_results[model_name] = cfg["all_result"]
            final_feature_sets[model_name] = X_train_scaled.columns.tolist()

final_train_metrics = pd.DataFrame(
    {
        model_name: result["train_metrics"]
        for model_name, result in final_results.items()
    }
).T
display(final_train_metrics.round(4))

final_best_params = pd.DataFrame(
    {
        model_name: pd.Series(result["grid"].best_params_)
        for model_name, result in final_results.items()
    }
).T
display(final_best_params)


## Визуализация дерева решений

Визуализируем итоговое дерево решений. Для читаемости ограничим глубину отображения значением `3`, но используем уже обученную лучшую модель.


In [ ]:
tree_features = final_feature_sets["DecisionTreeClassifier"]
tree_model = final_results["DecisionTreeClassifier"]["best_model"]

plt.figure(figsize=(24, 12))
plot_tree(
    tree_model,
    feature_names=tree_features,
    class_names=["No", "Yes"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=9,
)
plt.title("Итоговое дерево решений (показаны верхние уровни)")
plt.show()


## Сравнение лучших моделей на тестовой выборке

Ниже приводятся итоговые значения `accuracy`, `precision`, `recall`, `f1` на тестовой выборке. Именно этот этап используется для финального сравнения моделей.


In [ ]:
test_rows = []
for model_name, result in final_results.items():
    current_features = final_feature_sets[model_name]
    test_rows.append(
        test_result_row(
            model_name=model_name,
            model=result["best_model"],
            X_test=X_test_scaled[current_features],
            y_test=y_test,
        )
    )

test_metrics = (
    pd.DataFrame(test_rows)
    .set_index("model")
    .round(4)
    .sort_values(by="f1", ascending=False)
)
display(test_metrics)

best_model_name = test_metrics.index[0]
print(f"Лучшая модель на тестовой выборке по F1: {best_model_name}.")
print(
    "Итоговый выбор делается по F1, потому что для несбалансированной задачи "
    "важно одновременно контролировать precision и recall для класса 'Yes'."
)


## Итоговые выводы

В этой лабораторной работе:

- выполнена очистка и подготовка `weatherAUS.csv` для бинарной классификации `RainTomorrow`;
- для `DecisionTreeClassifier`, `RandomForestClassifier` и `KNeighborsClassifier` выполнен подбор гиперпараметров через `GridSearchCV`;
- для каждой модели проведено сравнение качества до и после фильтрации признаков;
- при наличии улучшения формируется общее подмножество признаков и модели переобучаются на одном наборе;
- финальный лидер определяется по метрикам на тестовой выборке.

Последняя ячейка выше автоматически покажет, какая модель стала лучшей после запуска ноутбука.
